# Corpus regression dataset

In [ ]:
from __future__ import annotations

from transformers import AutoTokenizer

from src import get_repo_base
from src.data.corpus_regression import (
    CorpusRegressionDataloadingConfig,
    CorpusRegressionDatasetConfig,
)

In [ ]:
cfg = CorpusRegressionDatasetConfig.get_canonical()
out_dir = get_repo_base() / "artifacts" / "corpus-regression" / cfg.get_canonical_folder_name()
cfg.visualize()

## 1. Build (or load cached)

`build_or_load` streams `HuggingFaceFW/fineweb-edu`, keeps docs of length ≥ `prefix_length + num_lookfoward_tokens`, takes their first `prefix_length` tokens as input, and assigns each sample the deterministic ±1 Rademacher embedding of the final lookforward token. Cached to `out_dir`; subsequent calls reload if the config matches.

In [ ]:
ds = cfg.build_or_load(out_dir)
print(f"train_tokens: {tuple(ds.train_tokens.shape)}  dtype={ds.train_tokens.dtype}")
print(f"train_labels: {tuple(ds.train_labels.shape)}  dtype={ds.train_labels.dtype}")
print(f"val_tokens:   {tuple(ds.val_tokens.shape)}")
print(f"val_labels:   {tuple(ds.val_labels.shape)}")
print(f"label range:  {ds.train_labels.min().item():.0f} … {ds.train_labels.max().item():.0f}")
print(f"label mean:   {ds.train_labels.mean().item():+.4f}  (≈0 expected)")

## 2. Inspect a sample

Decode a prefix back to text and print its label vector.

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.pretrained_tokenizer_model_name)
prefix_ids = ds.train_tokens[0].tolist()
print(f"prefix ({len(prefix_ids)} tokens):")
print(tok.decode(prefix_ids)[:400] + " …")
print(f"\nlabel ({ds.train_labels.shape[1]}-dim ±1):")
print(ds.train_labels[0].tolist())

## 3. Dataloaders

`CorpusRegressionDataloadingConfig` wraps each split in a `TensorDataset` and returns a `DataLoader`. Single-device by default; shuffle on for train, off for val.

In [ ]:
dl_cfg = CorpusRegressionDataloadingConfig(
    train_batch_size=256,
    eval_batch_size=512,
    drop_last=True,
)
train_dl = dl_cfg.get_train_dataloader(ds)
val_dl = dl_cfg.get_val_dataloader(ds)

batch_tokens, batch_labels = next(iter(train_dl))
print(f"batch tokens: {tuple(batch_tokens.shape)}  dtype={batch_tokens.dtype}")
print(f"batch labels: {tuple(batch_labels.shape)}  dtype={batch_labels.dtype}")
print(f"len(train_dl)={len(train_dl)}  len(val_dl)={len(val_dl)}")